# Gain with 95% confidence intervals

Mean participant gain and t-based confidence intervals across stimulus velocities.


In [1]:
import sys
from pathlib import Path

# Work whether Jupyter starts in the project root or in notebooks/.
search_roots = [Path.cwd(), *Path.cwd().parents]
project_root = next(
    (root for root in search_roots if (root / 'src' / 'config.py').is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        'Could not find the project root containing src/config.py. '
        'Open this notebook from inside the SmallRamp_Analysis project.'
    )
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import balancepy as bp
import balancepy.anaropia as bpa
import balancepy.anaropia_project as bpap

from src.config import paths

# Analysis settings


In [2]:
CONFIDENCE = 0.95
FREQUENCY_INDEX = 0

condition_groups = [
    ([1, 13], None),
    ([15, 16, 17], 0.03),
    ([4, 5], 0.05),
    (6, 0.15),
    (7, 0.25),
    (8, 0.5),
    (9, 1),
    (10, 5),
    (11, 10),
    ([2, 3, 14], 0.01),
    ([18, 19], 0.07),
    (20, 0.1),
    (21, 0.2),
    (22, 0.3),
]

preproc_config = bpa.AnaropiaPreprocessingConfig()
preproc_config.end_time_seconds = 210
preproc_config.cut_to_cycles = True
preproc_config.cycle_length_samples = 180
preproc_config.cycle_start_samples = 878

sr_config = bpa.AnaropiaSRDataConfig(
    stimulus_name='stim_pitch',
    response_name=bpa.COM_LEGACY_AP,
    frequency_selection='all',
)

# Load and preprocess data across all subjects

Keep participant IDs to calculate one gain value per participant.


In [3]:
data_by_velocity = []

for conditions, velocity in condition_groups:
    if velocity is None:
        continue

    print(f'Loading velocity {velocity} ...')
    records = []
    for subject_id, condition, filepath, body_height_m in bpap.batch_iterator(
        paths=paths,
        conditions=conditions,
        output_columns=['body_height_m'],
    ):
        sr = bpa.build_sr_data(
            filename=filepath,
            body_height_m=body_height_m,
            sr_config=sr_config,
            preproc_config=preproc_config,
            name=f'{subject_id}_{condition}',
        )
        records.append({'subject_id': subject_id, 'sr': sr})

    if not records:
        print(f'No usable data for velocity {velocity}; skipping it.')
        continue

    merged = bp.merge_sr_data([record['sr'] for record in records])
    data_by_velocity.append({
        'velocity': velocity,
        'records': records,
        'merged': merged,
    })
    print(f'  done ({len({record["subject_id"] for record in records})} participants)')

Loading velocity 0.03 ...


  done (12 participants)
Loading velocity 0.05 ...


  done (13 participants)
Loading velocity 0.15 ...


  done (13 participants)
Loading velocity 0.25 ...


  done (13 participants)
Loading velocity 0.5 ...


...

  done (13 participants)
Loading velocity 0.07 ...


  done (12 participants)
Loading velocity 0.1 ...


  done (12 participants)
Loading velocity 0.2 ...


  done (12 participants)
Loading velocity 0.3 ...


  done (12 participants)


# Mean participant gain with 95% confidence intervals

Calculate the arithmetic mean of participant gains and the t-based interval: $\mathrm{mean} \pm t\,s/\sqrt{n}$.


In [7]:

from scipy.stats import t

def participant_gains(records, frequency_index=0):
    """Return one pooled-within-participant gain for every participant."""
    contributions = {}

    for record in records:
        subject_id = record['subject_id']
        sr = record['sr']
        stimulus = np.atleast_2d(sr.stimulus_spectrum)
        response = np.atleast_2d(sr.response_spectrum)
        if stimulus.shape[0] == 1 and sr.stimulus_spectrum.ndim == 1:
            stimulus = stimulus.T
            response = response.T

        yi = stimulus[frequency_index]
        yo = response[frequency_index]
        numerator = np.sum(yo * np.conjugate(yi))
        denominator = np.sum(np.abs(yi) ** 2)

        if subject_id not in contributions:
            contributions[subject_id] = [0j, 0.0]
        contributions[subject_id][0] += numerator
        contributions[subject_id][1] += denominator

    return np.asarray([
        np.abs(numerator / denominator)
        for numerator, denominator in contributions.values()
    ])


t_results = []
for group in sorted(data_by_velocity, key=lambda item: item['velocity']):
    gains = participant_gains(group['records'], FREQUENCY_INDEX)
    n = len(gains)
    mean_gain = gains.mean()
    if n > 1:
        standard_error = gains.std(ddof=1) / np.sqrt(n)
        margin = t.ppf(1 - (1 - CONFIDENCE) / 2, df=n - 1) * standard_error
    else:
        margin = np.nan

    t_results.append({
        'velocity': group['velocity'],
        'mean_gain': mean_gain,
        'ci_lower': mean_gain - margin,
        'ci_upper': mean_gain + margin,
        'n_subjects': n,
    })

gain_t_ci = pd.DataFrame(t_results).sort_values('velocity').reset_index(drop=True)
gain_t_ci

,velocity,mean_gain,ci_lower,ci_upper,n_subjects
0,0.01,1.381582,0.788342,1.974823,13
1,0.03,0.477626,0.295943,0.659310,12
2,0.05,0.748236,0.483643,1.012828,13
3,0.07,0.568738,0.341005,0.796471,12
4,0.10,0.656115,0.402189,0.910041,12
5,0.15,0.546233,0.393389,0.699077,13
6,0.20,0.365594,0.254365,0.476823,12
7,0.25,0.441728,0.319198,0.564257,13
8,0.30,0.184682,0.104283,0.265081,12
9,0.50,0.193006,0.133783,0.252229,13


# Plot mean participant gain


In [8]:
fig_t = go.Figure()
# Thin background line showing the overall velocity order.
fig_t.add_trace(go.Scatter(
    x=gain_t_ci['velocity'],
    y=gain_t_ci['mean_gain'],
    mode='lines', line=dict(color='lightgrey', width=2),
    hoverinfo='skip', showlegend=False,
))

session_2_velocities = [0.01, 0.03, 0.07, 0.1, 0.2, 0.3]
session_2_mask = gain_t_ci['velocity'].isin(session_2_velocities)
session_1_mask = (~session_2_mask) | np.isclose(gain_t_ci['velocity'], 0.01)

for selection, color, session_name in (
    (session_1_mask, 'darkorange', 'Session 1'),
    (session_2_mask, 'darkgreen', 'Session 2'),
):
    plotted = gain_t_ci.loc[selection].sort_values('velocity')
    fig_t.add_trace(go.Scatter(
        x=plotted['velocity'],
        y=plotted['mean_gain'],
        mode='markers+lines',
        line=dict(color=color, width=2.2),
        marker=dict(color=color, size=8),
        name=session_name,
        customdata=np.column_stack([
            plotted['ci_lower'], plotted['ci_upper'], plotted['n_subjects']
        ]),
        error_y=dict(
            type='data', symmetric=False,
            array=plotted['ci_upper'] - plotted['mean_gain'],
            arrayminus=plotted['mean_gain'] - plotted['ci_lower'],
            thickness=1.5, width=5, color=color,
        ),
        hovertemplate=(
            'Velocity=%{x:g} °/s<br>'
            'Mean gain=%{y:.4g}<br>'
            '95% t-CI=[%{customdata[0]:.4g}, %{customdata[1]:.4g}]<br>'
            'Participants=%{customdata[2]:.0f}<extra></extra>'
        ),
    ))
fig_t.update_xaxes(
    type='log', title_text='Stimulus velocity (°/s)',
    showline=True, linecolor='black', linewidth=1,
    ticks='outside', tickfont=dict(size=15),
    showgrid=True, gridcolor='rgba(0, 0, 0, 0.10)', zeroline=False,
)
fig_t.update_yaxes(
    type='log', title_text='gain', title_standoff=12,
    showline=True, linecolor='black', linewidth=1,
    ticks='outside', tickfont=dict(size=15),
    showgrid=True, gridcolor='rgba(0, 0, 0, 0.10)', zeroline=False,
)
fig_t.update_layout(
    template='plotly_white',
    width=1150,
    height=520,
    font=dict(family='Times New Roman', size=16, color='black'),
    margin=dict(l=110, r=180, t=45, b=90),
    legend=dict(
        x=1.02, y=1.0, xanchor='left', yanchor='top',
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='lightgrey', borderwidth=1,
    ),
    plot_bgcolor='white', paper_bgcolor='white',
)
fig_t.show()

# Selected participant gain with 95% confidence intervals

Set `SELECTED_PARTICIPANT` below. Calculate the pooled gain and a leave-one-cycle-out jackknife-t interval at each velocity. The interval describes within-participant precision and assumes independent cycles.


In [9]:
# Change this participant code, then run this cell again.
SELECTED_PARTICIPANT = 'SI28IT27'


def original_gain_with_jackknife_t_ci(records, frequency_index=0, confidence=0.95):
    """Original pooled gain and a leave-one-cycle-out jackknife-t interval."""
    stimulus_cycles = []
    response_cycles = []

    for record in records:
        sr = record['sr']
        stimulus = sr.stimulus_spectrum
        response = sr.response_spectrum
        if stimulus.ndim == 1:
            stimulus = stimulus[:, None]
            response = response[:, None]
        stimulus_cycles.append(stimulus[frequency_index])
        response_cycles.append(response[frequency_index])

    yi = np.concatenate(stimulus_cycles)
    yo = np.concatenate(response_cycles)
    cross_power = yo * np.conjugate(yi)
    input_power = np.abs(yi) ** 2
    total_cross_power = cross_power.sum()
    total_input_power = input_power.sum()
    original_gain = np.abs(total_cross_power / total_input_power)
    n_cycles = len(yi)

    if n_cycles < 3:
        return original_gain, np.nan, np.nan, n_cycles

    leave_one_out_gain = np.abs(
        (total_cross_power - cross_power) / (total_input_power - input_power)
    )
    leave_one_out_gain = leave_one_out_gain[np.isfinite(leave_one_out_gain)]
    n_valid = len(leave_one_out_gain)
    if n_valid < 2:
        return original_gain, np.nan, np.nan, n_cycles

    jackknife_mean = leave_one_out_gain.mean()
    standard_error = np.sqrt(
        (n_valid - 1) / n_valid
        * np.sum((leave_one_out_gain - jackknife_mean) ** 2)
    )
    critical_t = t.ppf(1 - (1 - confidence) / 2, df=n_valid - 1)
    margin = critical_t * standard_error
    return original_gain, original_gain - margin, original_gain + margin, n_cycles


available_participants = sorted({
    str(record['subject_id'])
    for group in data_by_velocity
    for record in group['records']
})
if SELECTED_PARTICIPANT not in available_participants:
    raise ValueError(
        f'Unknown participant {SELECTED_PARTICIPANT!r}. '
        f'Choose one of: {available_participants}'
    )

selected_results = []
for group in sorted(data_by_velocity, key=lambda item: item['velocity']):
    selected_records = [
        record for record in group['records']
        if str(record['subject_id']) == SELECTED_PARTICIPANT
    ]
    if not selected_records:
        continue

    gain, ci_lower, ci_upper, n_cycles = original_gain_with_jackknife_t_ci(
        selected_records,
        frequency_index=FREQUENCY_INDEX,
        confidence=CONFIDENCE,
    )
    selected_results.append({
        'velocity': group['velocity'],
        'gain': gain,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'n_cycles': n_cycles,
    })

selected_gain_ci = pd.DataFrame(selected_results)

fig_selected = go.Figure()
fig_selected.add_trace(go.Scatter(
    x=selected_gain_ci['velocity'],
    y=selected_gain_ci['gain'],
    mode='markers+lines',
    name=SELECTED_PARTICIPANT,
    customdata=np.column_stack([
        selected_gain_ci['ci_lower'],
        selected_gain_ci['ci_upper'],
        selected_gain_ci['n_cycles'],
    ]),
    error_y=dict(
        type='data',
        symmetric=False,
        array=selected_gain_ci['ci_upper'] - selected_gain_ci['gain'],
        arrayminus=selected_gain_ci['gain'] - selected_gain_ci['ci_lower'],
        thickness=1.5,
        width=5,
    ),
    hovertemplate=(
        'velocity=%{x}<br>'
        'original gain=%{y:.4g}<br>'
        '95% CI=[%{customdata[0]:.4g}, %{customdata[1]:.4g}]<br>'
        'cycles=%{customdata[2]:.0f}<extra></extra>'
    ),
))
fig_selected.update_xaxes(type='log', title_text='velocity')
fig_selected.update_yaxes(type='log', title_text='gain')
fig_selected.update_layout(
    title=f'{SELECTED_PARTICIPANT}: original gain with 95% CI',
    template='plotly_white',
    width=900,
    height=600,
)
fig_selected.show()

selected_gain_ci

,velocity,gain,ci_lower,ci_upper,n_cycles
0,0.01,0.447339,-0.626365,1.521042,300
1,0.03,0.509306,-0.159375,1.177988,300
2,0.05,0.814996,0.404323,1.225669,200
3,0.07,0.550338,0.278160,0.822516,200
4,0.10,0.324876,0.037412,0.612341,100
5,0.15,0.921260,0.695308,1.147212,100
6,0.20,0.509654,0.239452,0.779856,100
7,0.25,0.486265,0.335129,0.637401,100
8,0.30,0.213872,0.090231,0.337514,100
9,0.50,0.203603,0.141652,0.265553,100


# Stimulus amplitude at 0.5 Hz


In [10]:
# Stimulus amplitude at the stimulation frequency versus stimulus velocity.
STIMULATION_FREQUENCY_HZ = 0.5
stimulus_amplitude_rows = []

for group in sorted(data_by_velocity, key=lambda item: item['velocity']):
    merged = group['merged']
    frequency_index = int(
        np.argmin(np.abs(merged.freq - STIMULATION_FREQUENCY_HZ))
    )
    actual_frequency = float(merged.freq[frequency_index])
    if not np.isclose(actual_frequency, STIMULATION_FREQUENCY_HZ):
        raise ValueError(
            f'Expected {STIMULATION_FREQUENCY_HZ} Hz but nearest bin is '
            f'{actual_frequency} Hz.'
        )

    stimulus_amplitude_rows.append({
        'velocity': group['velocity'],
        'stimulus_amplitude': float(
            np.abs(merged.stimulus_spectrum_mean[frequency_index])
        ),
    })

stimulus_amplitude = pd.DataFrame(stimulus_amplitude_rows)

fig_stimulus_amplitude = go.Figure()
fig_stimulus_amplitude.add_trace(go.Scatter(
    x=stimulus_amplitude['velocity'],
    y=stimulus_amplitude['stimulus_amplitude'],
    mode='markers+lines',
    line=dict(color='darkblue', width=2.2),
    marker=dict(color='darkblue', size=8),
    showlegend=False,
))
fig_stimulus_amplitude.update_xaxes(
    type='log', title_text='Stimulus velocity (°/s)',
    showline=True, linecolor='black', linewidth=1,
    ticks='outside', tickfont=dict(size=15),
    showgrid=True, gridcolor='rgba(0, 0, 0, 0.10)', zeroline=False,
)
fig_stimulus_amplitude.update_yaxes(
    type='log', title_text='Stimulus amplitude (°)', title_standoff=12,
    showline=True, linecolor='black', linewidth=1,
    ticks='outside', tickfont=dict(size=15),
    showgrid=True, gridcolor='rgba(0, 0, 0, 0.10)', zeroline=False,
)
fig_stimulus_amplitude.update_layout(
    template='plotly_white', width=1150, height=520,
    font=dict(family='Times New Roman', size=16, color='black'),
    margin=dict(l=110, r=45, t=45, b=90),
    plot_bgcolor='white', paper_bgcolor='white',
)
fig_stimulus_amplitude.show()

stimulus_amplitude

,velocity,stimulus_amplitude
0,0.01,0.004053
1,0.03,0.012159
2,0.05,0.020264
3,0.07,0.028370
4,0.10,0.040529
5,0.15,0.060793
6,0.20,0.081057
7,0.25,0.101323
8,0.30,0.121586
9,0.50,0.202642


# Sway response amplitude at 0.5 Hz


In [11]:
# Sway-response amplitude at 0.5 Hz versus stimulus velocity.
from plotly.subplots import make_subplots

STIMULATION_FREQUENCY_HZ = 0.5
sway_response_rows = []

for group in sorted(data_by_velocity, key=lambda item: item['velocity']):
    merged = group['merged']
    frequency_index = int(
        np.argmin(np.abs(merged.freq - STIMULATION_FREQUENCY_HZ))
    )
    actual_frequency = float(merged.freq[frequency_index])
    if not np.isclose(actual_frequency, STIMULATION_FREQUENCY_HZ):
        raise ValueError(
            f'Expected {STIMULATION_FREQUENCY_HZ} Hz but nearest bin is '
            f'{actual_frequency} Hz.'
        )

    sway_response_rows.append({
        'velocity': group['velocity'],
        'sway_response_amplitude': float(
            np.abs(merged.response_spectrum_mean[frequency_index])
        ),
    })

sway_response_amplitude = pd.DataFrame(sway_response_rows)

# No-stimulus recordings are separate conditions in the two sessions.
# Use the same pooled-merge estimator as the stimulated conditions; 0 °/s
# is never placed on the logarithmic stimulus-velocity axis.
no_stimulus_rows = []
for session_name, condition in (('Session 1', 1), ('Session 2', 13)):
    session_records = []
    for subject_id, _, filepath, body_height_m in bpap.batch_iterator(
        paths=paths, conditions=condition, output_columns=['body_height_m'],
    ):
        sr = bpa.build_sr_data(
            filename=filepath, body_height_m=body_height_m,
            sr_config=sr_config, preproc_config=preproc_config,
            name=f'{subject_id}_{condition}',
        )
        session_records.append(sr)

    pooled = bp.merge_sr_data(session_records)
    frequency_index = int(np.argmin(
        np.abs(pooled.freq - STIMULATION_FREQUENCY_HZ)
    ))
    mean_amplitude = float(np.abs(
        pooled.response_spectrum_mean[frequency_index]
    ))

    no_stimulus_rows.append({
        'session': session_name, 'mean_amplitude': mean_amplitude,
    })

no_stimulus_amplitude = pd.DataFrame(no_stimulus_rows)


In [12]:
# Keep no stimulus separate from the logarithmic velocity axis.
fig_sway_response_single = make_subplots(
    rows=1, cols=2, shared_yaxes=True, column_widths=[0.22, 0.78],
    horizontal_spacing=0.10,
)
linked_velocities = [0.01, 0.03, 0.07, 0.1, 0.2, 0.3]
linked_mask = sway_response_amplitude['velocity'].isin(linked_velocities)
other_mask = (~linked_mask) | np.isclose(
    sway_response_amplitude['velocity'], 0.01
)

for selection, color, session_name in (
    (other_mask, 'darkorange', 'Session 1'),
    (linked_mask, 'darkgreen', 'Session 2'),
):
    no_stimulus = no_stimulus_amplitude.query('session == @session_name').iloc[0]
    fig_sway_response_single.add_trace(go.Scatter(
        x=[session_name], y=[no_stimulus['mean_amplitude']], mode='markers',
        marker=dict(color=color, size=8), showlegend=False,
        hovertemplate=(
            '%{x}<br>Spontaneous sway amplitude=%{y:.4g} °<br>'
            '<extra></extra>'
        ),
    ), row=1, col=1)
    plotted = sway_response_amplitude.loc[selection].sort_values('velocity')
    fig_sway_response_single.add_trace(go.Scatter(
        x=plotted['velocity'],
        y=plotted['sway_response_amplitude'],
        mode='markers+lines',
        line=dict(color=color, width=2.2),
        marker=dict(color=color, size=8),
        name=session_name, showlegend=True,
        hovertemplate=(
            'Stimulus velocity=%{x:g} °/s<br>'
            'Sway response amplitude=%{y:.4g} °<extra></extra>'
        ),
    ), row=1, col=2)
fig_sway_response_single.update_xaxes(
    type='category', title_text='No visual stimulus',
    categoryorder='array', categoryarray=['Session 1', 'Session 2'],
    showline=True, linecolor='black', linewidth=1, ticks='outside',
    tickfont=dict(size=13), showgrid=False, zeroline=False, row=1, col=1,
)
fig_sway_response_single.update_xaxes(
    type='log', title_text='Stimulus velocity (°/s)',
    showline=True, linecolor='black', linewidth=1,
    ticks='outside', tickfont=dict(size=15),
    showgrid=True, gridcolor='rgba(0, 0, 0, 0.10)', zeroline=False,
    row=1, col=2,
)
fig_sway_response_single.update_yaxes(
    type='log', title_text='Spontaneous sway amplitude at 0.5 Hz (°)',
    title_standoff=12,
    showline=True, linecolor='black', linewidth=1,
    ticks='outside', tickfont=dict(size=15),
    showgrid=True, gridcolor='rgba(0, 0, 0, 0.10)', zeroline=False,
    row=1, col=1,
)
fig_sway_response_single.update_yaxes(
    type='log', title_text='Sway response amplitude at 0.5 Hz (°)',
    showticklabels=False, showline=True, linecolor='black',
    linewidth=1, ticks='outside',
    showgrid=True, gridcolor='rgba(0, 0, 0, 0.10)', zeroline=False,
    row=1, col=2,
)
fig_sway_response_single.update_layout(
    template='plotly_white', width=1150, height=520,
    font=dict(family='Times New Roman', size=16, color='black'),
    margin=dict(l=110, r=180, t=75, b=90),
    legend=dict(
        x=1.02, y=1.0, xanchor='left', yanchor='top',
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='lightgrey', borderwidth=1,
    ),
    plot_bgcolor='white', paper_bgcolor='white',
)
fig_sway_response_single.show(config={
    'toImageButtonOptions': {
        'format': 'svg',
        'filename': 'figure_sway_response_by_velocity',
        'width': 1150,
        'height': 520,
        'scale': 1,
    },
})

# Save thesis figures


In [13]:
# Save the final thesis figures as standalone interactive HTML files.
paper_figure_dir = paths.project_root / 'results' / 'paper_figures'
paper_figure_dir.mkdir(parents=True, exist_ok=True)

figures_to_export = {
    'figure_mean_participant_gain_t_ci': fig_t,
    'figure_stimulus_amplitude_by_velocity': fig_stimulus_amplitude,
    'figure_sway_response_by_velocity': fig_sway_response_single,
}

for filename, figure in figures_to_export.items():
    html_path = paper_figure_dir / f'{filename}.html'
    svg_path = paper_figure_dir / f'{filename}.svg'
    figure.write_html(
        html_path, include_plotlyjs=True, full_html=True,
        config={
            'responsive': True,
            'toImageButtonOptions': {
                'format': 'svg',
                'filename': filename,
                'width': figure.layout.width,
                'height': figure.layout.height,
                'scale': 1,
            },
        },
    )
    print(f'Saved: {html_path}')
    try:
        figure.write_image(svg_path, format='svg')
        print(f'Saved: {svg_path}')
    except (ImportError, ValueError) as error:
        print(f'SVG export skipped for {filename}: {error}')

print(
    'If SVG export was skipped, use the camera icon on the sway-response graph; '
    'its download format is configured as SVG.'
)

# Save mean participant gain table


In [14]:
# Export the data plotted in the t-based mean participant gain figure.
gain_table_dir = paths.project_root / 'results' / 'tables'
gain_table_dir.mkdir(parents=True, exist_ok=True)

gain_figure_table = gain_t_ci.copy()
gain_figure_table.insert(
    1, 'session',
    np.where(
        np.isclose(gain_figure_table['velocity'], 0.01),
        'Sessions 1 and 2',
        np.where(
            gain_figure_table['velocity'].isin(session_2_velocities),
            'Session 2', 'Session 1',
        ),
    ),
)
gain_figure_table = gain_figure_table.rename(columns={
    'velocity': 'Stimulus velocity (°/s)',
    'session': 'Session',
    'mean_gain': 'Mean participant gain',
    'ci_lower': '95% CI lower',
    'ci_upper': '95% CI upper',
    'n_subjects': 'Participants (n)',
})
gain_table_path = gain_table_dir / 'mean_participant_gain_t_ci.xlsx'
gain_figure_table.to_excel(gain_table_path, index=False)
print(f'Saved: {gain_table_path}')
gain_figure_table

,Stimulus velocity (°/s),Session,Mean participant gain,95% CI lower,95% CI upper,Participants (n)
0,0.01,Sessions 1 and 2,1.381582,0.788342,1.974823,13
1,0.03,Session 2,0.477626,0.295943,0.659310,12
2,0.05,Session 1,0.748236,0.483643,1.012828,13
3,0.07,Session 2,0.568738,0.341005,0.796471,12
4,0.10,Session 2,0.656115,0.402189,0.910041,12
5,0.15,Session 1,0.546233,0.393389,0.699077,13
6,0.20,Session 2,0.365594,0.254365,0.476823,12
7,0.25,Session 1,0.441728,0.319198,0.564257,13
8,0.30,Session 2,0.184682,0.104283,0.265081,12
9,0.50,Session 1,0.193006,0.133783,0.252229,13
